# **Acne Detection with YOLOV8**


### Import Dependencies


In [1]:
import os #buat buka file
import matplotlib.pyplot as plt #buat visualisasi
import cv2 #pustaka OpenCV (olah image)

# ini yang penting: Tensorflow
import tensorflow as tf #pustaka buat deep learning
from tensorflow.keras import * #ambil semua isi class nya
from tensorflow.keras.optimizers import Lion
from tensorflow.keras.optimizers import * #ambil semua isi class nya
import keras_cv #nanti YOLOV8 ada di sini

# Atur batch size
BATCH_SIZE = 32 #satu iterasi dia baca 32 image langsung
AUTO = tf.data.AUTOTUNE

d:\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
ERROR:absl:Detected mismatched Protobuf Gencode/Runtime major versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.5. Same major version is required. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "d:\python\Lib\site-packages\tensorflow_datasets\__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\python\Lib\site-packages\tensorflow_datasets\rlds\__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "d:\python\Lib\site-packages\tensorflow_datasets\rlds\envlogger_reader

### Preprocessing data

1. Baca data dan labelnya (karena image dan label terpisah)
2. Buat di list baru (isinya path image)

In [2]:
def parse_annotasi_txt(img_path, txt_path):
    '''
    Jadi dalam folder dataset tuh ada 2 folder. images = gambarnya, labels = annotasi koordinat acne dalam list [class_id posisi_atas_x posisi_atas_y lebar_%_dari_image tinggi_%_dari_gambar]
    Kurang lebih prosesnya kek gini:
    1. Buka gambar → tahu ukurannya.
    2. Buka file label → baca per baris (1 baris = 1 koordinat jerawat).
    3. Tiap baris: kalikan persentase dengan ukuran gambar, lalu hitung pojok kotaknya dari titik tengah.
    4. Kembalikan: path gambar, daftar kelas, daftar kotak.
    '''

    # baca gambar
    img = cv2.imread(img_path) #OpenCV tuh cuman bisa baca bentuk scalar, diubah dulu gambarnya ke scalar pake class imread ini
    h = int(img.shape[0]) # akses tinggi gambar
    w = int(img.shape[1]) # akses lebar gambar

    # baca label annotasi
    file_label = open(txt_path, 'r')

    # ubah anotasi(koordinat jerawat) ke satu string panjang yang dimasukin ke list
    lines = file_label.read().split('\n')

    #siapkan wadah hasil
    boxes = []
    classes = []

    #kalo baris pertama kosong, kembalikan nilai list yng kosong (gk ada jerawat)
    if lines[0] == '':
        return img_path, classes, boxes
    #kalo gk kosong, iterasi sebanyak jumlah string di list lines
    else:
        for i in range (0, len(lines)):
            obj = lines[i].split(' ') #ambil baris ke i, pecah berdasarkan spasi
            class_ = int(obj[0]) #ubah string pertama (yaitu ID) ke integer

            # ambil 4 angka sisanya (koordinat), ubah ke desimal
            x1 = float(obj[1])
            y1 = float(obj[2])
            w1 = float(obj[3])
            h1 = float(obj[4])

            # konversi ke piksel
            xmin = int((x1*w) - (w1*w)/2.0)
            ymin = int((y1*h) - (h1*h)/2.0)
            xmax = int((x1*w) + (w1*w)/2.0)
            ymax = int((y1*h) + (h1*h)/2.0)

            # simpan hasil
            boxes.append([xmin, ymin, xmax, ymax])
            classes.append(class_)

            # kembalikan hasil
            return img_path, classes, boxes

def create_path_list(path):
    '''
    path masing2 image nanti dibuat list, biar mudah dibaca/diakses
    '''
    full_path = []

    # variabel image nyimpen paths image -> sorted(urutin)
    images = sorted(os.listdir(path))

    # nama path masukin ke list, join dulu sama nama foldernya pake "/" (misal images/foto1.jpg)
    for i in images:
        full_path.append(os.path.join(path, i))
    
    return full_path

# tentukan class
class_ids = ['Acne']
class_mapping = {0: 'Acne'}

In [3]:
def create_files_dict(img_files_paths, annot_files_paths):
    '''
    Di sini functionnya berfungsi buat ngubah data ke bentuk dictionary
    '''

    img_files = create_path_list(img_files_paths)
    annot_files = create_path_list(annot_files_paths)
    
    image_paths = []
    bbox = []
    classes = []
    
    for i in range(0,len(img_files)):
        image_path_, classes_, bbox_ = parse_annotasi_txt(img_files[i], annot_files[i])
        image_paths.append(image_path_)
        bbox.append(bbox_)
        classes.append(classes_)
        
    image_paths = tf.ragged.constant(image_paths)
    bbox = tf.ragged.constant(bbox)
    classes = tf.ragged.constant(classes)
    
    return image_paths, classes, bbox

In [4]:
# gunain functionya

# 1. Data training
train_img_paths, train_classes, train_coord = create_files_dict(r'acne_dataset\acne\train\images', r'acne_dataset\acne\train\labels')

# 2. Data test
test_img_paths, test_classes, test_coord = create_files_dict(r'acne_dataset\acne\test\images', r'acne_dataset\acne\test\labels')

# 3. Data validation
valid_img_paths, valid_classes, valid_coord = create_files_dict(r'acne_dataset\acne\valid\images', r'acne_dataset\acne\valid\labels')

In [5]:
# hasil akhir bentukan datanya (paths)
train_img_paths[:5]

<tf.Tensor: shape=(5,), dtype=string, numpy=
array([b'acne_dataset\\acne\\train\\images\\acne-101_jpeg.rf.af45ba09bf2981ed7cc4af94b54aae48.jpg',
       b'acne_dataset\\acne\\train\\images\\acne-101_jpeg.rf.d13d618fff02c3133aac95c6b0c7b5fb.jpg',
       b'acne_dataset\\acne\\train\\images\\acne-101_jpeg.rf.dd5c14cef0a2ba13b110f05a520f00c0.jpg',
       b'acne_dataset\\acne\\train\\images\\acne-104_jpeg.rf.049f40b91f49dd3554b929ab004aa10e.jpg',
       b'acne_dataset\\acne\\train\\images\\acne-104_jpeg.rf.5b7dbce5a05426ce03d20ea30f8614ad.jpg'],
      dtype=object)>

In [6]:
# hasil akhir bentukan datanya (classes)
train_classes[:5]

<tf.RaggedTensor [[0],
 [0],
 [0],
 [0],
 [0]]>

In [7]:
# hasil akhir bentukan datanya (coord)
train_coord[:5]

<tf.RaggedTensor [[[422, 274, 456, 300]],

 [[274, 183, 300, 218]],

 [[422, 274, 456, 300]],

 [[313, 103, 377, 151]],

 [[313, 103, 377, 151]]]>

### Creating Pipeline Data
Ini dibuat sebagai jalur jalannya data ke model nanti

In [10]:
# Baca data image dan label sesuai permintaan model
def img_prepro(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3) #3 warna (RGB)
    img = tf.cast(img, tf.float32) #ubah tipe angka ke desimal float 32 bit

    return img

# inisialisasi resizer buat ubah ukuran sesuai permintaan YOLOV8
resizer = keras_cv.layers.JitteredResize(
    target_size=(640,640),
    scale_factor=(0.8,1.25),
    bounding_box_format='xyxy'
)

# fungsi load dataset
def loader(img_paths, classes, coord):
    img = img_prepro(img_paths)

    coordinate = {
        'classes': tf.cast(classes, dtype=tf.float32),
        'coord': coord
    }

    return {'images': img, 'coordinate': coordinate}

# ubah ke bentuk tuple
def dict_to_tuple(inputs):
    return inputs['images'], inputs['coordinate']

In [11]:
# Rangkai PIPELINE NYA guyss
# 1. Train datasets 
train_loader = tf.data.Dataset.from_tensor_slices((train_img_paths, train_classes, train_coord))
train_dataset = (train_loader.map(loader, num_parallel_calls=AUTO).shuffle(BATCH_SIZE*10).ragged_batch(BATCH_SIZE, drop_remainder=True).map(resizer, num_parallel_calls=AUTO).map(dict_to_tuple, num_parallel_calls=AUTO).prefetch(AUTO))

# 2. Valid datasets 
valid_loader = tf.data.Dataset.from_tensor_slices((valid_img_paths, valid_classes, valid_coord))
valid_dataset = (valid_loader.map(loader, num_parallel_calls=AUTO).shuffle(BATCH_SIZE*10).ragged_batch(BATCH_SIZE, drop_remainder=True).map(resizer, num_parallel_calls=AUTO).map(dict_to_tuple, num_parallel_calls=AUTO).prefetch(AUTO))

# 3. Test datasets 
test_loader = tf.data.Dataset.from_tensor_slices((test_img_paths, test_classes, test_coord))
test_dataset = (test_loader.map(loader, num_parallel_calls=AUTO).shuffle(BATCH_SIZE*10).ragged_batch(BATCH_SIZE, drop_remainder=True).map(resizer, num_parallel_calls=AUTO).map(dict_to_tuple, num_parallel_calls=AUTO).prefetch(AUTO))

In [12]:
# Visualisasikan sampel dataset akhirnya

def visualize_dataset(inputs, value_range, rows, cols, coordinate_format):
    input = next(iter(inputs.take(1)))
    images, coordinate = inputs[0], inputs[1] #masuk itu bentuk tuples images sama coordinatenya

    keras_cv.visualization.plot_bounding_box_gallery(
        images,
        value_range=value_range,
        rows=rows,
        cols=cols,
        y_true=coordinate,
        scale=6,
        font_scale=0.8,
        line_thickness=2,
        dpi=100,
        bounding_box_format=coordinate_format,
        class_mapping=class_mapping,
        true_color=(192,57,43)
    )

In [ ]:
# coba ambil dari data train
visualize_dataset(train_dataset, coordinate_format='xyxy', value_range=(0,255), rows=2, cols=2)